# HW3 Q4 [10 pts]



## Important Notices

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> add any cells to this Jupyter Notebook, because that will crash the autograder. Additionally, Make sure to delete or comment out any code you add to this notebook for testing purposes prior to submitting the notebook to Gradescope. Failure to do so may crash the autograder. 
</div>


All instructions, code comments, etc. in this notebook **are part of the assignment instructions**. That is, if there is instructions about completing a task in this notebook, that task is not optional.  



<div class="alert alert-block alert-info">
    You <strong>must</strong> implement the following functions in this notebook to receive credit.
</div>

`user()`

`load_data()`

`exclude_no_pickup_locations()`

`exclude_no_trip_distance()`

`include_fare_range()`

`get_highest_tip()`

`get_total_toll()`

Each method will be auto-graded using different sets of parameters or data, to ensure that values are not hard-coded.  You may assume we will only use your code to work with data from NYC Taxi Trips during auto-grading. You do not need to write code for unreasonable scenarios.  

Since the overall correctness of your code will require multiple function to work together correctly (i.e., all methods are interdepedent), implementing only a subset of the functions likely will lead to a low score.

### Helper functions

You should not use - and do not need - any helper functions; implement the required functions by completing the function skeletons below. All of the fuctions you implement should be self-contained; each function should carry all of the code it needs to execute on its expected input(s) within its body. 

### Kernel Selection

Be sure that you select the PySpark kernel from the kernel options above, if it is not already selected. Do not use the Python3 kernel. The notebook should default to the PySpark kernel. 

### GCP and Gradescope
For submitting to Gradescope, you should use spark dataframes and dataframe operations _only_. 
You should not use SQL. 

### SparkSession Objects and SparkContexts
Note that you have access to a SparkSession object in the notebook environment in virtue of using the PySpark kernel. It can be referenced using the variable name _spark_ and does not need to be explicitly created or defined. This beavior can be leveraged to read in the initial dataset. 

You should not need to invoke _spark_ in any of your functions but the load_data() function. Do not hardcode references to _spark_ into any of your functions except for the load_data() function. 

### Assumptions Regarding Function Inputs
This question assumes that you're carrying the results of your prior work foward at each stage of data preparation and analysis. Thus, the _input_ to each of the _filtering_ functions below should be the _output_ of the _prior_ function. The inputs to the two analytical functions should be the output of the last filtering function. 

#### Pyspark Imports
<span style="color:red">*Please don't modify the below cell*</span>

In [1]:
#export
import pyspark
import decimal

In [2]:
### Student Section - Please compete all the functions below

#### Function to return GT Username

In [3]:
#export
def user() -> str:
    """
    :return: string
    your GTUsername, NOT your 9-Digit GTId  
    """         
    return 'nheppermann3'

#### 4.2 - Function to load data

In [59]:
#export
from pyspark.sql.types import StructType, StructField, StringType, DecimalType
def load_data(gcp_storage_path: str) -> pyspark.sql.DataFrame:
    """
        :param gcp_storage_path: string (full gs path including file name e.g gs://bucket_name/data.csv) 
        :return: spark dataframe  
    """
    ################################################################
    # code to load yellow_tripdata_2019-01.csv data from your GCP  #
    # storage bucket                                               #      
    ################################################################
    
    # vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,
    # store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,
    # tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
    the_schema = StructType([
        StructField("vendorid", StringType(), True),
        StructField("tpep_pickup_datetime", StringType(), True),
        StructField("tpep_dropoff_datetime", StringType(), True),
        StructField("passenger_count", StringType(), True),
        StructField("trip_distance", DecimalType(38, 10), True),
        StructField("ratecodeid", StringType(), True),
        StructField("store_and_fwd_flag", StringType(), True),
        StructField("pulocationid", StringType(), True),
        StructField("dolocationid", StringType(), True),
        StructField("payment_type", StringType(), True), 
        StructField("fare_amount", DecimalType(38, 10), True),
        StructField("extra", StringType(), True),
        StructField("mta_tax", StringType(), True),
        StructField("tip_amount", DecimalType(38, 10), True),
        StructField("tolls_amount", DecimalType(38, 10), True),
        StructField("improvement_surcharge", StringType(), True),
        StructField("total_amount", StringType(), True),
        StructField("congestion_surcharge", StringType(), True) 
    ])
    df = spark.read.csv(
        gcp_storage_path, 
        header=True, 
        schema=the_schema
    )
    return df

#### 4.3 - Function to exclude trips that don't have a pickup location

In [60]:
#export
import pyspark.sql.functions as F
def exclude_no_pickup_locations(df: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
    """
        :param nyc tax trips dataframe: spark dataframe 
        :return: spark dataframe  
    """
    ################################################################
    # code to exclude trips with no pickup locations               #
    # Note: Exclude nulls and zeros                                #        
    ################################################################
    pulocationid_name = 'pulocationid'
    filtered_df = df.filter((F.col(pulocationid_name).isNotNull()) & (F.col(pulocationid_name) != 0))
    return filtered_df
    

#### 4.4 - Function to exclude trips with no distance

In [61]:
#export
def exclude_no_trip_distance(df: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
    """
        :param nyc tax trips dataframe: spark dataframe 
        :return: spark dataframe  
    """
    ################################################################
    # code to exclude trips with no trip distances                 #
    # Note: Exclude nulls and zeros                                #        
    ################################################################
    # trip distance column is null or zero
    trip_distance_name = 'trip_distance'
    filtered_df = df.filter((F.col(trip_distance_name).isNotNull()) & (F.col(trip_distance_name) != 0))
    return filtered_df    

#### 4.5 - Function to include fare amount between the range of 20 to 60 Dollars

In [62]:
#export
def include_fare_range(df: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
    
    """
        :param nyc tax trips dataframe: spark dataframe 
        :return: spark dataframe  
    """
    ################################################################
    # code to include trips with only within the fare range of     #
    # 20 to 60 dollars (including 20 and 60 dollars)               #        
    ################################################################
    #df.printSchema()
    fare_amount_name = 'fare_amount'
    filtered_df = df.filter(F.col(fare_amount_name).between(20.0, 60.0))
    #filtered_df = df.filter(((20.0 <= df[fare_amount_name]) & (df[fare_amount_name] <= 60.0)))
    #filtered_df = df.filter(((20.0 <= df[fare_amount_name]) & (df[fare_amount_name] <= 60.0)))
    return filtered_df
    
    

#### 4.6 - Function to get the highest tip amount

In [74]:
#export
from decimal import Decimal
def get_highest_tip(df: pyspark.sql.DataFrame) -> decimal.Decimal:
    """
        :param nyc tax trips dataframe: spark dataframe 
        :return: decimal (rounded to 2 digits)  (NOTE: DON'T USE FLOAT)
    """
    
    ################################################################
    # code to get the highest tip amount                           #
    #                                                              #        
    ################################################################
    tip_amount_name = 'tip_amount'
    #return df.orderBy(F.desc(tip_amount_name)).limit(1).select(tip_amount_name).first()[tip_amount_name]
    df = df.filter(F.col(tip_amount_name).isNotNull())
    val = Decimal(df.orderBy(F.desc(tip_amount_name)).select(tip_amount_name).first()[tip_amount_name])
    return round(val, 2)
    
    

#### 4.7 - Function to get total toll amount

In [87]:
#export
from decimal import Decimal
def get_total_toll(df: pyspark.sql.DataFrame) -> decimal.Decimal:
    """
        :param nyc tax trips dataframe: spark dataframe 
        :return: decimal (rounded to 2 digits)  (NOTE: DON'T USE FLOAT)
    """
    
    ################################################################
    # code to get total toll amount                                #
    #                                                              #        
    ################################################################
    tolls_amount_name = 'tolls_amount'
    total_toll_name = 'total_toll'
    df = df.filter(F.col(tolls_amount_name).isNotNull())
    val = Decimal(df.agg(F.sum(tolls_amount_name).alias(total_toll_name)).collect()[0][total_toll_name])
    return round(val, 2)

### Run above functions and print

#### Uncomment the cells below and test your implemented functions

#### Load data from yellow_tripdata09-08-2021.csv

In [88]:
gcp_storage_path = 'gs://nheppermann3/yellow_tripdata09-08-2021.csv'
df = load_data(gcp_storage_path)
df.printSchema()
df.show()

root
 |-- vendorid: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: decimal(38,10) (nullable = true)
 |-- ratecodeid: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: string (nullable = true)
 |-- dolocationid: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: decimal(38,10) (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: decimal(38,10) (nullable = true)
 |-- tolls_amount: decimal(38,10) (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+------------+------------+---------------------+------------+--------------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecodeid|store_and_fwd_flag|pulocationid|dolocationid|payment_type|  fare_amount|extra|mta_tax|  tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+------------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1| 1.5000000000|         1|                 N|         151|         239|           1| 7.0000000000|  0.5|    0.5|1.6500000000|0.0000000000|     

#### Print total numbers of rows in the dataframe

In [89]:
#fare_amount_name = 'fare_amount'
#df.summary("fare_amount").show()

In [90]:
df.count()

7667792

#### Print total number of rows in the dataframe after excluding trips with no pickup location

In [91]:
df_no_pickup_locations = exclude_no_pickup_locations(df)
df_no_pickup_locations.count()

3833896

#### Print total number of rows in the dataframe after exclude trips with no distance

In [92]:
df_no_trip_distance = exclude_no_trip_distance(df_no_pickup_locations)
df_no_trip_distance.count()

#summary_df = df_no_trip_distance.summary()

# Display the summary table
#summary_df.show()



2030274

#### Print total number of rows in the dataframe after including trips with fair amount between the range of 20 to 60 Dollars

In [93]:
fare_amount_name = 'fare_amount'
df_no_trip_distance.agg(F.min(fare_amount_name), F.max(fare_amount_name), F.avg(fare_amount_name)).show()

+----------------+-----------------+-----------------+
|min(fare_amount)| max(fare_amount)| avg(fare_amount)|
+----------------+-----------------+-----------------+
| -284.0000000000|623259.8600000000|12.47799791062684|
+----------------+-----------------+-----------------+



In [94]:
df_include_fare_range = include_fare_range(df_no_trip_distance)
print(df_include_fare_range.count())
df_include_fare_range.show()

255321


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+-------------+------------+---------------------+------------+--------------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecodeid|store_and_fwd_flag|pulocationid|dolocationid|payment_type|  fare_amount|extra|mta_tax|   tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+-------------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:45:21|  2019-01-01 01:31:05|              1| 8.7000000000|         1|                 N|         163|          25|           1|34.5000000000|  0.5|    0.5| 7.1500000000|0.0000000000| 

#### Print the highest tip amount

In [97]:
max_tip = get_highest_tip(df_include_fare_range)
#max_tip = get_highest_tip(df_no_trip_distance)
print(max_tip)

212.00


#### Print the total toll amount

In [98]:
total_toll = get_total_toll(df_include_fare_range)
#total_toll = get_total_toll(df_no_trip_distance)
print(total_toll)

NameError: name 'tolls_amount' is not defined